In [ ]:
import numpy as np
import pandas as pd
import warnings
import light_curve as lc
import optuna
import astropy

from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from tqdm import tqdm, TqdmWarning
from datetime import datetime
from pathlib import Path
from astropy.cosmology import Planck18
from scipy.optimize import curve_fit
from extinction import fitzpatrick99
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from typing import Literal, Callable, Any


warnings.filterwarnings("ignore", category=TqdmWarning)

In [ ]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


SEED = 67
EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [ ]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [ ]:
def load_all_feats_df(type: Type, **kwargs):
    return pd.concat([pd.read_parquet(filepath, **kwargs) for filepath in Path("../artifacts/feats").rglob(f"{type}_feats.parquet")])

def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __build_and_ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(total=len(log_df), unit="obj") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            pb.set_description(f"Building features for `{split}` (`{type}`)")

            feats_buf = []

            for obj_id, log_row in log_sub_df.iterrows():
                flc_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # pyright: ignore[reportArgumentType]
                feats = __build_feats_for_obj(log_row, flc_df)
                feats_buf.append(feats)

                pb.update()

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)

            pd.DataFrame(feats_buf, index=log_sub_df.index).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")


# TODO Add features acquired from domain knowledge
def __build_feats_for_obj(log_row: pd.Series, flc_df: pd.DataFrame) -> dict:
    # Could reorder?
    __de_extinct(log_row, flc_df)

    flc_df["Flux_ratio"] = flc_df["Flux"] / flc_df["Flux_err"]

    feats = log_row.to_dict()

    feats.update(__build_stats_feats_for_obj(flc_df))
    feats.update(__build_lc_feats_for_obj(flc_df))
    feats.update(__build_domain_feats_for_obj(flc_df))

    return feats


def __build_stats_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    feats = {}

    pivot_flc_df = flc_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    for agg_name, agg in __STATS_AGGS.items():
        for feat in ["Flux", "Flux_err", "Flux_ratio"]:
            feats[f"{feat}_{agg_name}"] = agg(flc_df[feat])
            
            for filter in __FILTERS:
                if filter in pivot_flc_df.columns:
                    feats[f"{feat}_{agg_name}_{filter}"] = agg(pivot_flc_df[(feat, filter)]) # pyright: ignore[reportCallIssue]

    return feats


__STATS_AGGS: dict[str, Callable[[pd.Series], Any]] = {
    "mean": np.mean,
    "std": np.std,
    "min": np.min,
    "max": np.max,
    "median": np.median,
    "q25": lambda feats: feats.quantile(0.25),
    "q75": lambda feats: feats.quantile(0.75),
}
__FILTERS = ["u", "g", "r", "i", "z", "y"]


def __build_lc_feats_for_obj(flc_df: pd.DataFrame) -> dict:   
    feats = {}

    def lc_fe(df: pd.DataFrame):
        return __LC_FE(df.index.to_numpy(dtype=np.float64), df["Flux"].to_numpy(), df["Flux_err"].to_numpy()) # pyright: ignore[reportCallIssue]

    # TODO Try optimizing
    flc_fin_df = flc_df[(np.isfinite(flc_df.index) & np.isfinite(flc_df["Flux"]) & np.isfinite(flc_df["Flux_err"]))]
    
    if len(flc_fin_df) >= __LC_FE_MIN_NROWS:
        lc_feats = lc_fe(flc_fin_df)

        for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
            feats[feat_name] = feat

    for filter in __FILTERS:
        flc_fin_sub_df = flc_fin_df.loc[flc_df["Filter"] == filter]

        if len(flc_fin_sub_df) >= __LC_FE_MIN_NROWS:
            lc_feats = lc_fe(flc_fin_sub_df)

            for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
                feats[f"{feat_name}_{filter}"] = feat

    return feats


__LC_FE = lc.Extractor(
    lc.LinearFit(), # pyright: ignore[reportArgumentType]
    lc.StetsonK(), # pyright: ignore[reportArgumentType]
    lc.Amplitude(), # pyright: ignore[reportArgumentType]
    lc.BeyondNStd(), # pyright: ignore[reportArgumentType]
    lc.Skew(), # pyright: ignore[reportArgumentType]
    lc.Kurtosis(), # pyright: ignore[reportArgumentType]
)
__LC_FE_MIN_NROWS = 4


# Effective Wavelengths (Angstroms) for LSST filters
# Source: MALLORN_using_data.ipynb
FILTER_WL = {
    'u': 3641.0, 
    'g': 4704.0, 
    'r': 6155.0, 
    'i': 7504.0, 
    'z': 8695.0, 
    'y': 10056.0
}

# --- Helper Functions ---

def get_dist_modulus(z: float) -> float:
    """Calculates distance modulus given redshift z."""
    if pd.isna(z) or z <= 0:
        return np.nan
    # Luminosity distance in parsecs
    d_l = Planck18.luminosity_distance(z).to(astropy.units.pc).value
    # Modulus = 5 * log10(d) - 5
    return 5 * np.log10(d_l + EPS) - 5

def planck_law(wave_angstrom, T):
    """
    Planck's law B_lambda(T).
    Source: TDEs are well described by a thermal blackbody[cite: 114, 146].
    """
    # Clip T to avoid overflow/underflow in exp
    T = np.clip(T, 1000, 1e6) 
    
    w_m = wave_angstrom * 1e-10
    h = 6.626e-34
    c = 3.0e8
    k = 1.38e-23
    
    # Calculate Intensity
    a = 2.0 * h * c**2
    b = h * c / (k * T)
    intensity = a / ( (w_m**5) * (np.exp(b / w_m) - 1.0) )
    return intensity

def power_law_model(t, amplitude, t0, baseline):
    """
    Theoretical TDE fallback model: L ~ (t - t0)^(-5/3)
    Source: Light curves follow a power-law decline[cite: 18, 269].
    """
    # Ensure t > t0 to avoid complex numbers/infinity
    dt = t - t0
    # Safe power law
    val = np.where(dt > 0, amplitude * np.power(dt, -5.0/3.0), 0)
    return val + baseline

# --- Core Functions ---

def __de_extinct(log_row: pd.Series, flc_sub_df: pd.DataFrame):
    """
    De-extincts flux using Fitzpatrick99 model.
    Source: MALLORN_using_data.ipynb
    Modifies flc_sub_df in-place.
    """
    ebv = log_row.get("EBV", 0.0)
    if pd.isna(ebv) or ebv < 0:
        ebv = 0.0

    # Get effective wavelengths for the filters present in this chunk
    # Using numpy array for vectorization with extinction lib
    filters = flc_sub_df["Filter"].values
    wave_eff = np.array([FILTER_WL.get(f, 5000.0) for f in filters])
    
    # Calculate extinction A_lambda
    # Rv is typically 3.1 for Milky Way
    a_lambda = fitzpatrick99(wave_eff, 3.1 * ebv)
    
    # Correction factor: Flux_true = Flux_obs * 10^(A / 2.5)
    correction_factor = np.power(10, a_lambda / 2.5)
    
    flc_sub_df["Flux"] *= correction_factor
    flc_sub_df["Flux_err"] *= correction_factor


def __build_domain_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    """
    Extracts domain-specific features for TDE classification.
    Requires that flc_df has been de-extincted and contains Z (redshift) 
    context if possible, or we infer it. 
    
    Note: Since the signature only accepts flc_df, we assume Z is not attached
    to flc_df rows in the user's snippet. However, for Rest-Frame calc, 
    we need Z. If Z is not in flc_df, we must rely on Observed Frame 
    or merge Z before calling this. 
    
    *Assumption*: The user calls this AFTER merging log_row info or 
    we treat timescale features in Observed Frame (suboptimal but robust).
    For this solution, I will perform robust fallback calculations.
    """
    feats = {}
    
    # Filter for valid measurements
    # Remove non-detections or negative fluxes for log-based features
    # Note: AGN can have negative flux [Domain Knowledge], but TDE peak fitting needs positive signal.
    valid_df = flc_df[flc_df['Flux'] > EPS].copy()
    
    if len(valid_df) < 3:
        return {k: np.nan for k in ['peak_flux_g', 'temp_bb_peak', 'decay_chi2']}

    # 1. IDENTIFY PEAK
    # Source: Luminosity 10^43.5-44.5 erg/s 
    peak_idx = valid_df['Flux'].idxmax()
    peak_row = valid_df.loc[peak_idx]
    t_peak = peak_row['Time (MJD)']
    feats['peak_flux_global'] = peak_row['Flux']
    
    # 2. COLOR FEATURES (AT PEAK)
    # Source: Persistent blue color (g-r < -0.1) [cite: 1156]
    # We take fluxes within a small window of the peak
    window = 10.0 # days
    peak_window = valid_df[np.abs(valid_df['Time (MJD)'] - t_peak) < window]
    
    mean_fluxes = peak_window.groupby('Filter')['Flux'].mean()
    
    if 'g' in mean_fluxes and 'r' in mean_fluxes:
        # Color: -2.5 log(Fg / Fr)
        ratio = mean_fluxes['g'] / (mean_fluxes['r'] + EPS)
        feats['color_g_r_peak'] = -2.5 * np.log10(ratio + EPS)
    else:
        feats['color_g_r_peak'] = np.nan

    if 'u' in mean_fluxes and 'g' in mean_fluxes:
        ratio = mean_fluxes['u'] / (mean_fluxes['g'] + EPS)
        feats['color_u_g_peak'] = -2.5 * np.log10(ratio + EPS)
    else:
        feats['color_u_g_peak'] = np.nan

    # 3. BLACKBODY TEMPERATURE FIT
    # Source: T ~ 10^4 - 10^5 K [cite: 19, 146]
    # We fit the SED constructed from mean fluxes at peak
    if len(mean_fluxes) >= 3:
        # Prepare arrays
        waves = np.array([FILTER_WL[f] for f in mean_fluxes.index])
        fluxes = mean_fluxes.values
        
        try:
            # Initial guess: 25,000 K (typical TDE)
            popt, _ = curve_fit(planck_law, waves, fluxes, p0=[25000.0], 
                                bounds=([1000.0], [200000.0]), maxfev=500)
            feats['temp_bb_peak'] = popt[0]
        except:
            feats['temp_bb_peak'] = np.nan
    else:
        feats['temp_bb_peak'] = np.nan

    # 4. LIGHTCURVE MORPHOLOGY (POWER LAW DECAY)
    # Source: Decline scales as t^-5/3 [cite: 18, 269]
    # We select the band with the most data points to fit the shape
    best_filter = flc_df['Filter'].value_counts().idxmax()
    band_lc = valid_df[valid_df['Filter'] == best_filter]
    
    # Only fit the decay (time > peak)
    decay_lc = band_lc[band_lc['Time (MJD)'] >= t_peak]
    
    if len(decay_lc) >= 5:
        t_data = decay_lc['Time (MJD)'].values
        y_data = decay_lc['Flux'].values
        
        # Normalize for stability
        t_norm = t_data - t_peak + 1.0 # Start at t=1
        y_norm = y_data / (np.max(y_data) + EPS)
        
        try:
            # Fit: y = A * t^(-5/3) + B
            # Fix t0 approx to 0 (since we shifted t_norm)
            popt_pl, _ = curve_fit(
                lambda t, a, b: power_law_model(t, a, 0, b),
                t_norm, y_norm, 
                p0=[1.0, 0.0], bounds=([0, -np.inf], [np.inf, np.inf]),
                maxfev=1000
            )
            
            # Calculate Residuals / Chi2
            y_pred = power_law_model(t_norm, *popt_pl, 0)
            residuals = y_norm - y_pred
            feats['decay_chi2_powerlaw'] = np.sum(residuals**2) / len(residuals)
            
        except:
            feats['decay_chi2_powerlaw'] = np.nan
    else:
        feats['decay_chi2_powerlaw'] = np.nan

    # 5. SMOOTHNESS (Von Neumann Ratio)
    # Source: TDEs are smooth vs AGN stochasticity [cite: 229]
    if len(band_lc) > 4:
        fluxes = band_lc.sort_values('Time (MJD)')['Flux'].values
        diffs = np.diff(fluxes)
        mean_diff_sq = np.mean(diffs**2)
        var = np.var(fluxes) + EPS
        feats['smoothness_vn'] = mean_diff_sq / var
    else:
        feats['smoothness_vn'] = np.nan

    return feats


# # TODO Consider extinction.fitzpatrick99
# def __de_extinct(log_row: pd.Series, flc_sub_df: pd.DataFrame):
#     r_λ = flc_sub_df["Filter"].map({
#         "u": 4.81,
#         "g": 3.64,
#         "r": 2.70,
#         "i": 2.06,
#         "z": 1.58,
#         "y": 1.31
#     })

#     c_λ = np.pow(10, 0.4 * r_λ * log_row["EBV"])

#     flc_sub_df["Flux"] *= c_λ
#     flc_sub_df["Flux_err"] *= c_λ


__build_and_ingest_feats(type="train")
__build_and_ingest_feats(type="test")


### Data Cleaning

In [ ]:
train_df = load_all_feats_df(type="train")

X = train_df.drop(columns=["SpecType", "English Translation", "split", "target"])
y = train_df["target"]

X

### Hyperparameter Tuning with Manual F1 Threshold Optimization

In [ ]:
# def __calc_best_f1_score_and_threshold(y: pd.Series, probs):
#     thresholds = np.linspace(0.01, 0.99, 1000)
#     f1_scores = np.array([f1_score(y, (probs > threshold).astype(int)) for threshold in thresholds])

#     idx = f1_scores.argmax()

#     return f1_scores[idx], thresholds[idx]


__SKF = StratifiedKFold(n_splits=10, random_state=SEED, shuffle=True)
__OPTUNA_SAMPLER = optuna.samplers.TPESampler(seed=SEED, multivariate=True)

In [ ]:
def __cb_objective(trial: optuna.Trial):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC", # More smooth convergence than F1
        "verbose": False,
        "allow_writing_files": False,
        "task_type": "CPU", # CPU is faster for 3k rows & required for 'Ordered'
        
        # Crucial for small data (prevents gradient leakage)
        "boosting_type": "Ordered", 
        "bootstrap_type": "Bernoulli", 
        
        # Handle Imbalance
        "auto_class_weights": "Balanced",
        
        # Tuning
        "iterations": trial.suggest_int("cb_iterations", 100, 1000),
        "learning_rate": trial.suggest_float("cb_learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("cb_depth", 3, 8), # Keep shallow for small data
        "l2_leaf_reg": trial.suggest_float("cb_l2_leaf_reg", 1, 10, log=True),
        "subsample": trial.suggest_float("cb_subsample", 0.5, 0.95),
        "random_strength": trial.suggest_float("cb_random_strength", 1e-9, 10, log=True),
    }

    scores = []

    for train_idx, val_idx in __SKF.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        cb_clf = CatBoostClassifier(**params)
        cb_clf.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
        
        probs = cb_clf.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, probs)
        scores.append(score)

    return np.mean(scores)


cb_study = optuna.create_study(direction="maximize", sampler=__OPTUNA_SAMPLER)
cb_study.optimize(__cb_objective, n_trials=200, n_jobs=-1, show_progress_bar=True) # pyright: ignore[reportArgumentType]

In [ ]:
def __brf_objective(trial: optuna.Trial):
    params = {
        "n_estimators": trial.suggest_int("brf_n_estimators", 100, 800),
        "max_depth": trial.suggest_int("brf_max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("brf_min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("brf_max_features", ["sqrt", "log2", None]),
        "sampling_strategy": "all", # Force resampling to achieve balance
        "replacement": True,
        "n_jobs": -1,
        "random_state": SEED,
    }

    scores = []

    for train_idx, val_idx in __SKF.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        brf_clf = BalancedRandomForestClassifier(**params)
        brf_clf.fit(X_train, y_train)
        
        probs = brf_clf.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, probs)
        scores.append(score)

    return np.mean(scores)


brf_study = optuna.create_study(direction="maximize", sampler=__OPTUNA_SAMPLER)
brf_study.optimize(__brf_objective, n_trials=200, n_jobs=-1, show_progress_bar=True) # pyright: ignore[reportArgumentType]

In [ ]:
best_cb_params = cb_study.best_params
best_cb_params.update({
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "verbose": False,
    "allow_writing_files": False,
    "task_type": "CPU",
    "boosting_type": "Ordered", 
    "bootstrap_type": "Bernoulli", 
    "auto_class_weights": "Balanced",
})

best_brf_params = brf_study.best_params
best_brf_params.update({
    "sampling_strategy": "all",
    "replacement": True,
    "n_jobs": -1,
    "random_state": SEED,
})

cb_oof = np.zeros(len(X))
brf_oof = np.zeros(len(X))

for train_idx, val_idx in __SKF.split(X, y):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    cb_clf = CatBoostClassifier(**best_cb_params)
    cb_clf.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
    cb_oof[val_idx] = cb_clf.predict_proba(X_val)[:, 1]

    brf_clf = BalancedRandomForestClassifier(**best_brf_params)
    brf_clf.fit(X_train, y_train)
    brf_oof[val_idx] = brf_clf.predict_proba(X_val)[:, 1]


def find_best_ensemble_vectorized(y, cb_probs, brf_probs, n_weights=1001, n_thresholds=10000):
    """
    Finds the optimal weight and threshold simultaneously using 3D broadcasting.
    
    Dimensions Legend:
    - T: Number of Thresholds (axis 0)
    - W: Number of Weights (axis 1)
    - N: Number of Data points (axis 2)
    """
    
    # 1. Define Grids
    # Weights from 0.0 to 1.0
    weights = np.linspace(0.0, 1.0, n_weights) # Shape: (W,)
    # Thresholds from 0.01 to 0.99
    thresholds = np.linspace(0.01, 0.99, n_thresholds) # Shape: (T,)

    # 2. Create Weighted Probabilities Matrix
    # Broadcasting: (W, 1) * (N,) -> (W, N)
    # Result: A matrix where every row is a different blend of the models
    p_weighted = (weights[:, None] * cb_probs) + ((1 - weights[:, None]) * brf_probs)

    # 3. Create 3D Prediction Tensor
    # We compare (1, W, N) against (T, 1, 1) to generate a (T, W, N) boolean tensor
    # This tensor contains the binary prediction for every single point, 
    # for every weight, for every threshold.
    preds_tensor = p_weighted[None, :, :] >= thresholds[:, None, None]

    # 4. Vectorized Confusion Matrix Calculation
    # Align y_true to (1, 1, N) for broadcasting
    y_broad = y[None, None, :].astype(bool)
    
    # Sum over axis 2 (the data points) to get counts for each (Threshold, Weight) pair
    # Logical AND is fast on booleans
    tp = (preds_tensor & y_broad).sum(axis=2)
    fp = (preds_tensor & ~y_broad).sum(axis=2)
    fn = (~preds_tensor & y_broad).sum(axis=2)

    # 5. Compute F1 Grid (T, W)
    # F1 = 2TP / (2TP + FP + FN)
    denominator = 2 * tp + fp + fn
    
    # Safe division to handle cases where model predicts all 0s (denom=0)
    f1_grid = np.divide(
        2 * tp, 
        denominator, 
        out=np.zeros_like(denominator, dtype=float), 
        where=denominator != 0
    )

    # 6. Find the Absolute Maximum
    # argmax gives the flattened index; unravel_index converts it back to (T, W)
    best_idx = np.unravel_index(np.argmax(f1_grid), f1_grid.shape)
    best_t_idx, best_w_idx = best_idx

    return {
        "best_f1": f1_grid[best_idx],
        "best_weight_catboost": weights[best_w_idx], # Weight for p_catboost
        "best_weight_brf": 1 - weights[best_w_idx],  # Weight for p_brf
        "best_threshold": thresholds[best_t_idx]
    }

results = find_best_ensemble_vectorized(y, cb_oof, brf_oof)

print(f"Max F1 Score: {results['best_f1']:.5f}")
print(f"Optimal Mix : {results['best_weight_catboost']:.2f} (CatBoost) / {results['best_weight_brf']:.2f} (BRF)")
print(f"Threshold   : {results['best_threshold']:.4f}")

### Feature Importance Analysis

### Inference

In [ ]:
test_df = load_all_feats_df(type="test")

X_test = test_df.drop(columns=["SpecType", "English Translation", "split"])

X_test

In [ ]:
cb_clf = CatBoostClassifier(**best_cb_params)
cb_clf.fit(X, y, verbose=False)

brf_clf = BalancedRandomForestClassifier(**best_brf_params)
brf_clf.fit(X, y)

In [ ]:
probs_cb = cb_clf.predict_proba(X_test)[:, 1]
probs_brf = brf_clf.predict_proba(X_test)[:, 1]

w_cb = results['best_weight_catboost']
w_brf = results['best_weight_brf']

ensemble_probs = (w_cb * probs_cb) + (w_brf * probs_brf)

best_threshold = results['best_threshold']
test_preds = (ensemble_probs >= best_threshold).astype(int)

In [ ]:
Path("../artifacts/preds").mkdir(parents=True, exist_ok=True)

test_preds_df = pd.DataFrame({"object_id": X_test.index, "target": test_preds})
test_preds_df.to_csv(f"../artifacts/preds/submission-{now()}.csv", index=False)

In [ ]:
# TODO
# Assuming GB, no imputation, no scaling, only minimal cleaning.
# Plot f1 score changes over trials & hyperparams
# Plot feature importance
# Set seeds to 67